# Stage 4, tier 3: full-transcript half-life and translation efficiency, behind their gate

Research plan **Priority 3** (full-transcript half-life) and **Priority 4** (translation
efficiency), both Gantt 42. The plan is explicit that these are gated, not assumed:

> **Feasibility gate.** Verify sequences include 5'/3' UTRs, CDS coordinates, labels, species
> and non-leaking train/test groups. If transcript regions are absent, do not treat it as a
> full-transcript test.

> **Gate.** Confirm the measured target (TE or mean ribosome load), full transcript / UTR /
> CDS availability, experimental context and valid grouped splits. Do not conflate TE with
> protein abundance.

So Part 1 is the gate, run on the published files themselves, and it decides what Parts 2 and
3 are allowed to claim. The gate is cheap: a few minutes and about 150 MB of downloads, no
GPU and no model. **Run Part 1 before committing a GPU night to Part 3.**

### Why this task exists at all

The stability benchmark BioLangFusion uses is coding sequence only. Stage 4 Phase A found
that its published split repeats training sequences in 68% of test rows, and that on the rows
it does not repeat, every model drops by 0.2 or more. A clean transcript-level half-life task
answers a question that benchmark cannot: **does the sequence outside the coding region carry
anything a codon-level model is missing**, and does fusion still help once it is there.

### The three questions, kept apart

The plan is emphatic that these labels are not interchangeable, and mislabelling them is the
most likely way to produce a wrong result that looks right:

| Target | What it measures | Dataset here |
| --- | --- | --- |
| RNA half-life | degradation rate of endogenous transcripts | `rnahl-human`, `rnahl-mouse` |
| Translation efficiency | ribosome profiling over transcript abundance | `translation-efficiency-human`, `-mouse` |
| Mean ribosome load | ribosomes per transcript isoform, **not** half-life | `mrl-sugimoto` |

`mrl-hl-lbkwk` carries one half-life target and one ribosome-load target on synthetic mRNAs,
and they are run separately. Nothing in this notebook pools two of these.

---

## Setup

In [1]:
# Standard library
import json
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code import analysis, encoders, mrnabench, sequences
from shared_code.encoders import ENCODERS

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| Apple GPU:", torch.backends.mps.is_available())

torch 2.10.0+cu126 | CUDA: True | Apple GPU: False


In [8]:
RESULTS_DIR = Path("results") / "halflife_te"
EMBED_DIR = Path("../stage3/embeddings") / "mrnabench"
DATA_CACHE = mrnabench.DEFAULT_CACHE          # .data-cache/mrnabench, outside Git

# The two datasets Part 3 embeds. Human first; the mouse pair is the replication.
PRIMARY = ("rnahl-human", "translation-efficiency-human")
PROBE_SEEDS = (2541, 2547, 413, 412, 411, 321, 421, 2515, 2516)   # mRNABench's own nine
RUN_EMBEDDINGS = True                 # Part 3 only; read the cost projection first

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"device {DEVICE}; results in {RESULTS_DIR}, embeddings in {EMBED_DIR}")

device cuda; results in results\halflife_te, embeddings in ..\stage3\embeddings\mrnabench


## Part 1: the gate

Every number below is measured from the published file, not read from its description. The
coding region is not given as coordinates: each row carries a binary track marking the first
nucleotide of every codon, so the region is recovered from the marks and the spacing between
them is what proves the frame. A row whose marks are not three apart does not describe a
reading frame and is counted rather than quietly dropped.

The downloads are cached in `.data-cache/`, which Git ignores.

In [3]:
gate_rows, frames = [], {}
for key, spec in mrnabench.DATASETS.items():
    started = time.perf_counter()
    frames[key] = mrnabench.load(key, DATA_CACHE)
    gate_rows.append(mrnabench.gate(frames[key], spec))
    print(f"  {key}: {len(frames[key]):,} rows in {time.perf_counter() - started:.0f} s")

gate = pd.DataFrame(gate_rows).set_index("dataset")
display(gate[["rows", "species", "targets present", "rows with a coding region",
              "rows with a 5' region", "rows with a 3' region", "median transcript nt",
              "longest transcript nt", "distinct genes"]]
        .style.format({"rows": "{:,}", "rows with a coding region": "{:.1%}",
                       "rows with a 5' region": "{:.1%}", "rows with a 3' region": "{:.1%}",
                       "median transcript nt": "{:,}", "longest transcript nt": "{:,}",
                       "distinct genes": "{:,}"}))

  rnahl-human: 12,939 rows in 144 s
  rnahl-mouse: 13,723 rows in 33 s
  translation-efficiency-human: 10,281 rows in 20 s
  translation-efficiency-mouse: 10,853 rows in 38 s
  mrl-sugimoto: 11,693 rows in 17 s
  mrl-hl-lbkwk: 203 rows in 1 s


,rows,species,targets present,rows with a coding region,rows with a 5' region,rows with a 3' region,median transcript nt,longest transcript nt,distinct genes
dataset,,,,,,,,,
rnahl-human,"12,939",human,target 100%,100.0%,100.0%,99.6%,"3,105","12,288","12,939"
rnahl-mouse,"13,723",mouse,target 100%,100.0%,100.0%,99.7%,"2,872","12,288","13,723"
translation-efficiency-human,"10,281",human,target 100%,100.0%,100.0%,100.0%,"3,391","17,497","10,280"
translation-efficiency-mouse,"10,853",mouse,target 100%,100.0%,100.0%,100.0%,"3,109","12,252","10,848"
mrl-sugimoto,"11,693",human,target 100%,84.9%,98.6%,89.4%,"1,976","12,275","7,392"
mrl-hl-lbkwk,203,synthetic,"target_in_cell_half_life 100%, target_ribosome_load 100%",100.0%,100.0%,100.0%,925,"1,585",0


In [4]:
for key, row in gate.iterrows():
    print(f"{key}\n    verdict: {row['verdict']}")
    if row["caveats"]:
        print(f"    watch:   {row['caveats']}")
    print(f"    measures: {row['measures']}")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
gate.to_csv(RESULTS_DIR / "gate.csv")

rnahl-human
    verdict: passes: full transcript, regions and grouped splits all present
    watch:   0.4% of coding rows have no 3' region
    measures: transcript half-life, first principal component over 39 human datasets
rnahl-mouse
    verdict: passes: full transcript, regions and grouped splits all present
    watch:   0.3% of coding rows have no 3' region
    measures: transcript half-life, first principal component over 27 mouse datasets
translation-efficiency-human
    verdict: passes: full transcript, regions and grouped splits all present
    measures: translation efficiency from ribosome profiling, one transcript per gene
translation-efficiency-mouse
    verdict: passes: full transcript, regions and grouped splits all present
    measures: translation efficiency from ribosome profiling, one transcript per gene
mrl-sugimoto
    verdict: passes: full transcript, regions and grouped splits all present
    watch:   15.1% of rows mark no coding region, so a region comparison mus

### What the gate decides

Read the verdicts above before going on. The rules this notebook then follows:

* A dataset whose rows carry both untranslated regions and a reading frame **may** be used
  for the region comparison of Part 2 and the fusion comparison of Part 3.
* A dataset without them is a sequence-level task and is reported as one. It does not answer
  Priority 3's question.
* Rows marking no coding region are **excluded from the region comparison and counted**, not
  silently dropped: on an isoform-resolved dataset they are the non-coding isoforms, and
  their share is itself a fact about the dataset.
* A dataset of a few hundred rows is kept out of the headline comparisons, because a grouped
  split of it leaves a test set too small to resolve the differences being compared.

In [5]:
usable = [key for key, row in gate.iterrows() if row["verdict"].startswith("passes")
          and gate.loc[key, "rows"] >= mrnabench.MINIMUM_ROWS]
print("datasets that pass the gate and are large enough:")
for key in usable:
    print(f"  {key}: {mrnabench.DATASETS[key].measures}")
held_back = [k for k in mrnabench.DATASETS if k not in usable]
print("\nheld back from the headline comparison:")
for key in held_back:
    print(f"  {key}: {gate.loc[key, 'verdict']} | {gate.loc[key, 'caveats']}")

datasets that pass the gate and are large enough:
  rnahl-human: transcript half-life, first principal component over 39 human datasets
  rnahl-mouse: transcript half-life, first principal component over 27 mouse datasets
  translation-efficiency-human: translation efficiency from ribosome profiling, one transcript per gene
  translation-efficiency-mouse: translation efficiency from ribosome profiling, one transcript per gene
  mrl-sugimoto: mean ribosome load per transcript isoform, NOT half-life

held back from the headline comparison:
  mrl-hl-lbkwk: passes: full transcript, regions and grouped splits all present | 203 rows is too few to resolve small differences; the library is synthetic, so rows are split at random and a gene grouping does not apply


## Part 2: how much of the transcript matters, before any foundation model

The plan asks for "simple length and composition baselines" beside the encoders, and they
come first for a reason: if composition of the whole transcript already predicts half-life,
then an encoder has to beat that rather than beat nothing. Each region is scored on its own
with the same probe and the same grouped split, so the regions are compared on equal terms.

This part needs no GPU and no model: it is ridge on counted features.

In [6]:
# The region sets the plan names. They are nested: "full transcript" carries the coding
# sequence's own features plus the two untranslated regions, so comparing the two rows asks
# whether the regions outside the coding sequence add anything.
REGION_SETS = {
    "CDS only": ("cds",),
    "CDS + 3' UTR": ("cds", "utr3"),
    "5' UTR only": ("utr5",),
    "3' UTR only": ("utr3",),
    "full transcript": ("utr5", "cds", "utr3"),
}
print("features per set, for one example row:")
example = mrnabench.region_frame(mrnabench.load(PRIMARY[0], DATA_CACHE).head(2))
for name, parts in REGION_SETS.items():
    print(f"  {name:<18} {mrnabench.region_features(example, parts).shape[1]} features")

features per set, for one example row:
  CDS only           89 features
  CDS + 3' UTR       91 features
  5' UTR only        2 features
  3' UTR only        2 features
  full transcript    93 features


In [7]:
baseline_rows = []
for key in usable:
    spec = mrnabench.DATASETS[key]
    regions = mrnabench.region_frame(frames[key])
    keep = regions["frame_problem"] == ""
    regions = regions[keep].reset_index(drop=True)
    groups, grouping_described = mrnabench.grouping(regions, spec.species, DATA_CACHE)
    print(f"\n{key}: {len(regions):,} rows with a reading frame, grouped by {grouping_described}")
    for target in spec.targets:
        y = regions[target].to_numpy(dtype=float)
        for name, parts in REGION_SETS.items():
            X = mrnabench.region_features(regions, parts)
            scores = mrnabench.probe_over_seeds(X, y, groups, PROBE_SEEDS[:3])
            baseline_rows.append({"dataset": key, "target": target, "representation": name,
                                  "test r": scores["test_r"].mean(),
                                  "spread": scores["test_r"].std(),
                                  "seeds": len(scores)})
            print(f"    {name:<18} Pearson r {scores['test_r'].mean():.3f} "
                  f"(+/- {scores['test_r'].std():.3f} over {len(scores)} splits)")

baselines = pd.DataFrame(baseline_rows)
baselines.to_csv(RESULTS_DIR / "composition_baselines.csv", index=False)
display(baselines.pivot_table(index=["dataset", "target"], columns="representation",
                              values="test r").round(3))


rnahl-human: 12,939 rows with a reading frame, grouped by gene (the homology map is not downloaded; weaker than the benchmark's)
    CDS only           Pearson r 0.487 (+/- 0.003 over 3 splits)
    CDS + 3' UTR       Pearson r 0.534 (+/- 0.007 over 3 splits)
    5' UTR only        Pearson r 0.177 (+/- 0.017 over 3 splits)
    3' UTR only        Pearson r 0.260 (+/- 0.018 over 3 splits)
    full transcript    Pearson r 0.542 (+/- 0.016 over 3 splits)

rnahl-mouse: 13,723 rows with a reading frame, grouped by gene (the homology map is not downloaded; weaker than the benchmark's)
    CDS only           Pearson r 0.447 (+/- 0.021 over 3 splits)
    CDS + 3' UTR       Pearson r 0.487 (+/- 0.023 over 3 splits)
    5' UTR only        Pearson r 0.173 (+/- 0.010 over 3 splits)
    3' UTR only        Pearson r 0.254 (+/- 0.022 over 3 splits)
    full transcript    Pearson r 0.498 (+/- 0.020 over 3 splits)

translation-efficiency-human: 10,277 rows with a reading frame, grouped by gene (the homo

,representation,3' UTR only,5' UTR only,CDS + 3' UTR,CDS only,full transcript
dataset,target,,,,,
mrl-sugimoto,target,0.031,0.035,0.198,0.200,0.199
rnahl-human,target,0.260,0.177,0.534,0.487,0.542
rnahl-mouse,target,0.254,0.173,0.487,0.447,0.498
translation-efficiency-human,target,0.135,0.280,0.685,0.683,0.702
translation-efficiency-mouse,target,0.088,0.234,0.632,0.629,0.660


## Part 3: the encoders, and whether fusion still helps on a whole transcript

The comparison the plan asks for, in its own words: 5' UTR, CDS, full transcript, full
transcript with a protein representation, and the three-encoder fusion, with the gain
reported **over the best full-transcript baseline** rather than over a weak regional input.

Two things make this more expensive than Phase A. The transcripts are long, a median of
about 3,000 nucleotides against the benchmarks' 999, and there are around 12,000 of them per
dataset. And every encoder except the k-mer ones has a context shorter than one transcript,
so a transcript is embedded in consecutive windows whose token states are concatenated before
pooling, which is mRNABench's own rule and is recorded as a deviation below.

The cell after next measures the cost before anything starts.

In [9]:
# One encoder per modality, the same three BioLangFusion uses, so the comparison carries over.
STREAM_ENCODERS = {"dna": "ntv2_100m", "rna": "rnafm", "prot": "esm2_8m"}
INPUTS = {
    "5' UTR": ("rna", ("utr5",)),
    "CDS": ("rna", ("cds",)),
    "full transcript": ("rna", ("utr5", "cds", "utr3")),
    "full transcript, DNA encoder": ("dna", ("utr5", "cds", "utr3")),
    "protein of the CDS": ("prot", ("cds",)),
}

for key in PRIMARY:
    if key not in usable:
        print(f"{key} did not pass the gate; Part 3 skips it.")
print(f"\nwindows per encoder: " + ", ".join(
    f"{k}: {mrnabench.default_window(ENCODERS[v]):,} characters"
    for k, v in STREAM_ENCODERS.items()))


windows per encoder: dna: 12,276 characters, rna: 1,022 characters, prot: 1,022 characters


In [10]:
def stream_texts(regions, modality, parts):
    # The strings one encoder reads for a region set: nucleotides in its own alphabet, or the
    # translation of the coding sequence for a protein encoder.
    encoder = ENCODERS[STREAM_ENCODERS[modality]]
    if modality == "prot":
        return [mrnabench.encoder_text(encoder, row["cds"]) for _, row in regions.iterrows()]
    joined = ["".join(row[p] for p in parts) for _, row in regions.iterrows()]
    return [mrnabench.encoder_text(encoder, text) for text in joined]


def cost_projection(regions, seconds_per_1000_nt=0.02):
    # A rough projection from total nucleotides, refined by the timing cell below.
    total = sum(len(row["utr5"]) + len(row["cds"]) + len(row["utr3"])
                for _, row in regions.iterrows())
    return total / 1000 * seconds_per_1000_nt / 3600

In [11]:
if RUN_EMBEDDINGS:
    embedded = {}
    for key in PRIMARY:
        spec = mrnabench.DATASETS[key]
        regions = mrnabench.region_frame(frames[key])
        regions = regions[regions["frame_problem"] == ""].reset_index(drop=True)
        for name, (modality, parts) in INPUTS.items():
            texts = stream_texts(regions, modality, parts)
            folder = EMBED_DIR / key
            tag = name.replace("'", "").replace(" ", "_").replace(",", "")
            started = time.perf_counter()
            matrix, reused = mrnabench.embed_with_cache(
                STREAM_ENCODERS[modality], texts, folder / f"{tag}.npz", device=DEVICE)
            embedded[(key, name)] = matrix
            print(f"  {key} / {name}: {matrix.shape}, "
                  f"{'reused' if reused else f'{(time.perf_counter() - started) / 60:.1f} min'}")
else:
    print("RUN_EMBEDDINGS is off. Part 1 and Part 2 run without it; turn it on for a GPU night.")

W1022 05:32:32.914000 8888 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: 200/12939
    rnafm: 400/12939
    rnafm: 600/12939
    rnafm: 800/12939
    rnafm: 1000/12939
    rnafm: 1200/12939
    rnafm: 1400/12939
    rnafm: 1600/12939
    rnafm: 1800/12939
    rnafm: 2000/12939
    rnafm: 2200/12939
    rnafm: 2400/12939
    rnafm: 2600/12939
    rnafm: 2800/12939
    rnafm: 3000/12939
    rnafm: 3200/12939
    rnafm: 3400/12939
    rnafm: 3600/12939
    rnafm: 3800/12939
    rnafm: 4000/12939
    rnafm: 4200/12939
    rnafm: 4400/12939
    rnafm: 4600/12939
    rnafm: 4800/12939
    rnafm: 5000/12939
    rnafm: 5200/12939
    rnafm: 5400/12939
    rnafm: 5600/12939
    rnafm: 5800/12939
    rnafm: 6000/12939
    rnafm: 6200/12939
    rnafm: 6400/12939
    rnafm: 6600/12939
    rnafm: 6800/12939
    rnafm: 7000/12939
    rnafm: 7200/12939
    rnafm: 7400/12939
    rnafm: 7600/12939
    rnafm: 7800/12939
    rnafm: 8000/12939
    rnafm: 8200/12939
    rnafm: 8400/12939
    rnafm: 8600/12939
    rnafm: 8800/12939
    rnafm: 9000/12939
    rnafm: 920

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

...

[transformers] RnaFmModel LOAD REPORT from: multimolecule/rnafm
Key                                 | Status     | 
------------------------------------+------------+-
ss_head.decoder.bias                | UNEXPECTED | 
lm_head.transform.dense.weight      | UNEXPECTED | 
lm_head.transform.layer_norm.weight | UNEXPECTED | 
ss_head.decoder.weight              | UNEXPECTED | 
lm_head.transform.dense.bias        | UNEXPECTED | 
lm_head.transform.layer_norm.bias   | UNEXPECTED | 
lm_head.bias                        | UNEXPECTED | 
pooler.dense.bias                   | MISSING    | 
pooler.dense.weight                 | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    rnafm: 200/10277
    rnafm: 400/10277
    rnafm: 600/10277
    rnafm: 800/10277
    rnafm: 1000/10277
    rnafm: 1200/10277
    rnafm: 1400/10277
    rnafm: 1600/10277
    rnafm: 1800/10277
    rnafm: 2000/10277
    rnafm: 2200/10277
    rnafm: 2400/10277
    rnafm: 2600/10277
    rnafm: 2800/10277
    rnafm: 3000/10277
    rnafm: 3200/10277
    rnafm: 3400/10277
    rnafm: 3600/10277
    rnafm: 3800/10277
    rnafm: 4000/10277
    rnafm: 4200/10277
    rnafm: 4400/10277
    rnafm: 4600/10277
    rnafm: 4800/10277
    rnafm: 5000/10277
    rnafm: 5200/10277
    rnafm: 5400/10277
    rnafm: 5600/10277
    rnafm: 5800/10277
    rnafm: 6000/10277
    rnafm: 6200/10277
    rnafm: 6400/10277
    rnafm: 6600/10277
    rnafm: 6800/10277
    rnafm: 7000/10277
    rnafm: 7200/10277
    rnafm: 7400/10277
    rnafm: 7600/10277
    rnafm: 7800/10277
    rnafm: 8000/10277
    rnafm: 8200/10277
    rnafm: 8400/10277
    rnafm: 8600/10277
    rnafm: 8800/10277
    rnafm: 9000/10277
    rnafm: 920

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t6_8M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    esm2_8m: 200/10277
    esm2_8m: 400/10277
    esm2_8m: 600/10277
    esm2_8m: 800/10277
    esm2_8m: 1000/10277
    esm2_8m: 1200/10277
    esm2_8m: 1400/10277
    esm2_8m: 1600/10277
    esm2_8m: 1800/10277
    esm2_8m: 2000/10277
    esm2_8m: 2200/10277
    esm2_8m: 2400/10277
    esm2_8m: 2600/10277
    esm2_8m: 2800/10277
    esm2_8m: 3000/10277
    esm2_8m: 3200/10277
    esm2_8m: 3400/10277
    esm2_8m: 3600/10277
    esm2_8m: 3800/10277
    esm2_8m: 4000/10277
    esm2_8m: 4200/10277
    esm2_8m: 4400/10277
    esm2_8m: 4600/10277
    esm2_8m: 4800/10277
    esm2_8m: 5000/10277
    esm2_8m: 5200/10277
    esm2_8m: 5400/10277
    esm2_8m: 5600/10277
    esm2_8m: 5800/10277
    esm2_8m: 6000/10277
    esm2_8m: 6200/10277
    esm2_8m: 6400/10277
    esm2_8m: 6600/10277
    esm2_8m: 6800/10277
    esm2_8m: 7000/10277
    esm2_8m: 7200/10277
    esm2_8m: 7400/10277
    esm2_8m: 7600/10277
    esm2_8m: 7800/10277
    esm2_8m: 8000/10277
    esm2_8m: 8200/10277
    esm2_8m: 8400/10

In [12]:
if RUN_EMBEDDINGS and embedded:
    rows_out = []
    for key in PRIMARY:
        spec = mrnabench.DATASETS[key]
        regions = mrnabench.region_frame(frames[key])
        regions = regions[regions["frame_problem"] == ""].reset_index(drop=True)
        groups, described = mrnabench.grouping(regions, spec.species, DATA_CACHE)
        for target in spec.targets:
            y = regions[target].to_numpy(dtype=float)
            available = {name: embedded[(key, name)] for _, name in
                         [(k, n) for (k, n) in embedded if k == key]}
            available["full transcript + protein"] = np.hstack(
                [available["full transcript"], available["protein of the CDS"]])
            available["three encoders fused"] = np.hstack(
                [available["full transcript, DNA encoder"], available["full transcript"],
                 available["protein of the CDS"]])
            for name, X in available.items():
                scores = mrnabench.probe_over_seeds(X, y, groups, PROBE_SEEDS)
                rows_out.append({"dataset": key, "target": target, "representation": name,
                                 "test r": scores["test_r"].mean(),
                                 "standard error": scores["test_r"].std() / np.sqrt(len(scores)),
                                 "test Spearman": scores["test_spearman"].mean(),
                                 "seeds": len(scores), "grouping": described})
                print(f"  {key} / {name}: r {scores['test_r'].mean():.3f}")
    encoder_scores = pd.DataFrame(rows_out)
    encoder_scores.to_csv(RESULTS_DIR / "encoder_probes.csv", index=False)
    display(encoder_scores.pivot_table(index=["dataset", "target"], columns="representation",
                                       values="test r").round(3))
else:
    print("Nothing to score yet.")

  rnahl-human / 5' UTR: r 0.178
  rnahl-human / CDS: r 0.473
  rnahl-human / full transcript: r 0.512
  rnahl-human / full transcript, DNA encoder: r 0.514
  rnahl-human / protein of the CDS: r 0.552
  rnahl-human / full transcript + protein: r 0.606
  rnahl-human / three encoders fused: r 0.602
  translation-efficiency-human / 5' UTR: r 0.433
  translation-efficiency-human / CDS: r 0.630
  translation-efficiency-human / full transcript: r 0.610
  translation-efficiency-human / full transcript, DNA encoder: r 0.660
  translation-efficiency-human / protein of the CDS: r 0.682
  translation-efficiency-human / full transcript + protein: r 0.712
  translation-efficiency-human / three encoders fused: r 0.713


,representation,5' UTR,CDS,full transcript,full transcript + protein,"full transcript, DNA encoder",protein of the CDS,three encoders fused
dataset,target,,,,,,,
rnahl-human,target,0.178,0.473,0.512,0.606,0.514,0.552,0.602
translation-efficiency-human,target,0.433,0.630,0.610,0.712,0.660,0.682,0.713


In [13]:
# The plan's definition, applied: the gain over the best FULL-TRANSCRIPT baseline, not over a
# weak regional input. Both the best single encoder and the composition floor are included,
# because a fusion that beats one and not the other has not earned the claim.
if RUN_EMBEDDINGS and "encoder_scores" in dir() and len(encoder_scores):
    gains = []
    for (key, target), block in encoder_scores.groupby(["dataset", "target"]):
        by_name = block.set_index("representation")["test r"]
        singles = [n for n in ("full transcript", "full transcript, DNA encoder",
                               "protein of the CDS") if n in by_name]
        best_single = by_name[singles].max()
        floor = baselines.query("dataset == @key and target == @target and "
                                "representation == 'full transcript'")["test r"]
        for fused in ("full transcript + protein", "three encoders fused"):
            if fused in by_name:
                gains.append({
                    "dataset": key, "target": target, "fused model": fused,
                    "fused r": by_name[fused], "best single r": best_single,
                    "delta fusion": by_name[fused] - best_single,
                    "over composition": by_name[fused] - (floor.iloc[0] if len(floor) else np.nan),
                })
    display(pd.DataFrame(gains).round(3))
    pd.DataFrame(gains).to_csv(RESULTS_DIR / "fusion_gains.csv", index=False)

,dataset,target,fused model,fused r,best single r,delta fusion,over composition
0,rnahl-human,target,full transcript + protein,0.606,0.552,0.053,0.064
1,rnahl-human,target,three encoders fused,0.602,0.552,0.050,0.060
2,translation-efficiency-human,target,full transcript + protein,0.712,0.682,0.030,0.010
3,translation-efficiency-human,target,three encoders fused,0.713,0.682,0.031,0.012


## What this notebook may and may not conclude

* A positive gain here is **incremental predictive benefit under this protocol**, as the plan
  says, not evidence of independent biological information. The probe, the split and the
  pooling are all fixed across rows of the table, and that is the whole of the claim.
* The absolute values are not comparable with mRNABench's published table: that benchmark
  reports its own models under its own homology splits, and uses chunked embeddings for
  models whose context is short, which this notebook also does but with different encoders.
* Half-life, translation efficiency and mean ribosome load are three different measurements.
  Every table above is keyed by dataset and target so that none of them can be read as
  another.

## Deviations recorded for this notebook

| # | deviation | why |
| --- | --- | --- |
| 1 | **Grouping by gene rather than by the benchmark's homology groups**, unless the homology map has been downloaded into `.data-cache/mrnabench/homology_maps/` | the map is a separate download; grouping by gene is strictly weaker, because paralogues can still sit on both sides, so this is recorded rather than passed off as the benchmark's own split |
| 2 | A transcript longer than an encoder's context is embedded in consecutive windows, and the token states are concatenated before pooling | mRNABench's own rule for short-context models; a window cannot see outside itself, so a chunked embedding is an approximation of the whole transcript rather than the thing itself |
| 3 | Frozen encoders with a ridge probe, where RiboNN and Saluki train end to end | the project's premise is frozen fusion, and the plan asks for the gain over the best full-transcript baseline under one protocol, not for a new state of the art |
| 4 | Nine split seeds in Part 3, three in Part 2 | Part 2 compares five region sets across several datasets and targets, where three splits already separate them; Part 3 reports the differences the project will quote, and uses the benchmark's own nine |
| 5 | Rows marking no coding region are excluded from the region comparison | they have no CDS to compare against, and their share is reported in the gate instead |
| 6 | ESM-2 8M as the protein encoder, not a larger one | it is BioLangFusion's own protein encoder, so a gain here is comparable with Phase A's |